# scRareBench high-level template for any integration method

Use this notebook when your method is **not** one of the example methods in the repository. You control dependency installation, preprocessing, training, seed handling, and latent extraction. scRareBench only consumes the latent and evaluates it.

The default PCA block is a runnable wiring smoke test, **not a batch-correction method**. Replace `run_user_method` for a real benchmark.


In [ ]:
import subprocess, sys
REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@v0.10.5"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO])

# Edit these for your method. Empty means no extra method package is installed.
METHOD_DEPENDENCIES = ()
METHOD_IMPORTS = ()
INSTALL_METHOD_DEPENDENCIES = False

from scrarebench.runtime import setup_runtime
setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=METHOD_IMPORTS if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


In [ ]:
from scrarebench import load_dataset, dataset_info
DATASET = 0
adata = load_dataset(DATASET)
info = dataset_info(adata)
print(info)


In [ ]:
METHOD_NAME = "PCA_smoke_test_not_batch_corrected"
METHOD_SEEDS = [42]             # e.g. [42, 123, 2026] for multi-seed
BENCHMARK_SEED = 42
METHOD_CONFIG = {"n_components": 30}

# For a custom dataset instead of a registered dataset:
# from scrarebench import register_dataset
# register_dataset(adata, label_key="cell_type", batch_key="batch", rare_types=["RareType"])


## Replace only this method runner

Required signature: `runner(method_adata, seed, config)`. Return a NumPy array/DataFrame, an `obsm` key, or `MethodOutput`. If your method reorders cells, return barcodes in `MethodOutput`.


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from scrarebench import MethodOutput

def run_user_method(method_adata, seed, config):
    # RUNNABLE SMOKE TEST ONLY. Replace this body with your integration method.
    X = method_adata.X.toarray() if hasattr(method_adata.X, "toarray") else np.asarray(method_adata.X)
    n_components = min(int(config["n_components"]), X.shape[0] - 1, X.shape[1])
    latent = PCA(n_components=n_components, random_state=int(seed)).fit_transform(X)
    return MethodOutput(latent=latent, barcodes=method_adata.obs_names)


In [ ]:
from scrarebench import MethodSpec, benchmark_method
method = MethodSpec(
    name=METHOD_NAME,
    runner=run_user_method,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)
result = benchmark_method(
    adata, method,
    seeds=METHOD_SEEDS,
    benchmark_config={"random_state": BENCHMARK_SEED},
    install_dependencies=False,
    finalize=True,
)
print("Report:", result.report_path)
print("Archive:", result.archive_path)
